# Run NVIDIA Nemotron Models on Nebius Token Factory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nebius/token-factory-cookbook/blob/main/models/nemotron/run_nemotron_colab.ipynb)
[![](https://img.shields.io/badge/Powered%20by-Nebius-orange?style=flat&labelColor=darkblue&color=orange)](http://tokenfactory.nebius.com/)

NVIDIA Nemotron is a family of open mixture-of-experts models, ranging from fast, low-cost models for always-on agents and high-throughput tasks up to flagship models for the most demanding multi-agent and reasoning workloads.

This notebook works with any Nemotron model on Token Factory. Set `MODEL_NAME` below to the one you want to use:

| Model ID | Parameters | Context |
|----------|------------|---------|
| `nvidia/Nemotron-3_5-Lightning` | 30B total / 3B active | 262K |
| `nvidia/nemotron-3-super-120b-a12b` | 120B total / 12B active | 256K |
| `nvidia/Nemotron-3-Ultra-550b-a55b` | 550B total / 55B active | 256K |

See the full list at [tokenfactory.nebius.com/models](https://tokenfactory.nebius.com/models).


## 1 - Getting Started

### 1.1 - Get your Nebius API key at [Nebius Token Factory](https://tokenfactory.nebius.com/)

### 1.2 - If running on Google Colab

Add `NEBIUS_API_KEY` to the **Secrets** panel on the left.

![](https://github.com/nebius/token-factory-cookbook/raw/main/images/google-colab-1.png)

### 1.3 - If running locally

Create a `.env` file in this directory with your key:

```text
NEBIUS_API_KEY=your_api_key_goes_here
```


## 2 - Install Dependencies

If you set up this directory with `uv sync`, the dependencies are already installed and you can skip this cell.


In [ ]:
!pip install -q openai python-dotenv

## 3 - Load Configuration

In [ ]:
import os

if os.getenv("COLAB_RELEASE_TAG"):
    print("Running in Colab")
    from google.colab import userdata
    NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")
else:
    print("NOT running in Colab")
    from dotenv import load_dotenv
    load_dotenv()
    NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY")

if NEBIUS_API_KEY:
    print("✅ NEBIUS_API_KEY found")
    os.environ["NEBIUS_API_KEY"] = NEBIUS_API_KEY
else:
    raise RuntimeError("❌ NEBIUS_API_KEY not found. See section 1 above.")

## 4 - Run the Model

### 4.1 - Create a client


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

# Pick any Nemotron model available on Token Factory
MODEL_NAME = "nvidia/Nemotron-3_5-Lightning"
# MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"
# MODEL_NAME = "nvidia/Nemotron-3-Ultra-550b-a55b"

### 4.2 - Ask a simple question

In [ ]:
%%time

completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant"},
        {"role": "user", "content": "What is the capital of France?"},
    ],
    temperature=0.6,
)

print("---- model answer ----")
print(completion.choices[0].message.content)
print("\n---- usage ----")
print(completion.usage)

### 4.3 - Stream a longer answer

In [ ]:
%%time

stream = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": "Write a short Python function that reads a CSV file and prints the number of rows."},
    ],
    temperature=0.6,
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        print(delta, end="", flush=True)
print()

## 5 - Try Your Own Queries

Some ideas to get started:

> Which is bigger, 9.9 or 9.11?

> Write a haiku about GPUs

> Summarize the plot of Hamlet in three sentences
